# SafeCityAI custom YOLOv5 — training and handover

This notebook is **unexecuted scaffolding**, not an executed notebook result.
Use Runtime → Change runtime type → GPU. The published draft branch includes a
checksum-pinned public-data pilot with genuine Helmet / NoHelmet / LicensePlate
annotations, explicit polygon conversion and whole-source-group quarantine.
Alternatively supply your own licensed, independently split dataset.

The small public subset is experimental: it is not sufficient evidence for
operational enforcement. Read `docs/PUBLIC_DATA.md` before training. No automatic
production installation or deployment is performed by this notebook.


In [ ]:
from pathlib import Path
import subprocess, sys, os, json

ROOT = Path("/content/safecityai")
REPO = "https://github.com/akshaydip11-source/object-detection-yolov5-traffic.git"
REF = "arena/01a0cfd1-object-detection-yolov5-traffi"
if not ROOT.exists():
    subprocess.run(["git", "clone", "--branch", REF, REPO, str(ROOT)], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(ROOT / "requirements.txt")],
    check=True,
)
import torch

print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Enable a Colab GPU before the full training run.")

## Choose the reviewed public pilot or your own licensed dataset
The default imports the reviewed public export and retains attribution/provenance.
It does not infer helmet status from generic person/motorbike labels. The import
fails closed on unknown classes/licenses, excessive exclusions or missing classes.
Set `DATA_SOURCE = "drive"` to use your own data instead. Split by original
camera/video before extracting frames; never put private footage or credentials in Git.


In [ ]:
import yaml

DATA_SOURCE = "public-pilot"  # Or "drive" for your own licensed dataset.
if DATA_SOURCE == "public-pilot":
    DATASET = ROOT / "runs/public-colab-data"
    subprocess.run(
        [sys.executable, "-m", "training.public_dataset", "--output", str(DATASET),
         "--polygon-boxes", "--quarantine-invalid"],
        cwd=ROOT, check=True,
    )
    DATA = DATASET / "data.yaml"
    print((DATASET / "ATTRIBUTION.md").read_text())
elif DATA_SOURCE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    DATASET = Path("/content/drive/MyDrive/traffic-dataset")  # Edit this path.
    if not DATASET.is_dir():
        raise FileNotFoundError("Supply the licensed labeled dataset before continuing.")
    DATA = ROOT / "outputs/colab-data.yaml"
    DATA.parent.mkdir(parents=True, exist_ok=True)
    config = {"path": str(DATASET), "train": "images/train", "val": "images/val",
              "names": {0: "Helmet", 1: "NoHelmet", 2: "LicensePlate"}}
    if (DATASET / "images/test").is_dir():
        config["test"] = "images/test"
    DATA.write_text(yaml.safe_dump(config))
else:
    raise ValueError("Choose public-pilot or drive")
subprocess.run(
    [sys.executable, "training/train_yolov5.py", "--data", str(DATA), "--validate-only"],
    cwd=ROOT, check=True,
)


## Train and evaluate — do not install automatically
Adjust batch size for GPU memory. Training downloads the official YOLOv5s
initialization and learns from the validated annotations. Pretrained initialization
alone is not the deliverable. This GPU run differs from the smaller CPU pilot;
its accuracy must be measured independently, never copied from someone else's model.


In [ ]:
subprocess.run(
    [
        sys.executable,
        "training/train_yolov5.py",
        "--data",
        str(DATA),
        "--weights",
        "yolov5s.pt",
        "--epochs",
        "50",
        "--batch",
        "16",
        "--device",
        "0",
    ],
    cwd=ROOT,
    check=True,
)
reports = sorted((ROOT / "runs/train").glob("safecity-*/training_report.json"))
report = json.loads(reports[-1].read_text())
print(json.dumps(report, indent=2))
BEST = Path(report["checkpoint"])


## Review real metrics and held-out examples
Inspect the actual results.csv, loss plots, confusion matrices and validation
outputs. Training completion does not imply acceptable precision/recall/mAP.
A small curated public split does not establish target-camera performance.
Supply a representative independent image and traffic video below. The public
subset does not supply an independently validated temporal video; do not replace
it with an animated still and call that real-world validation.


In [ ]:
IMAGE = DATASET / "heldout/example.jpg"  # Supply real held-out files.
VIDEO = DATASET / "heldout/example.mp4"
subprocess.run(
    [
        sys.executable,
        "-m",
        "scripts.verify_release",
        "--image",
        str(IMAGE),
        "--video",
        str(VIDEO),
    ],
    cwd=ROOT,
    check=True,
    env={**os.environ, "MODEL_PATH": str(BEST)},
)

## Download the handover artifacts
Only run after reviewing the actual evaluation and held-out smoke output.
Keep weights/data out of Git; transfer the artifact through trusted storage.
SHA256 identifies the file, not its accuracy. Preserve the public dataset's
ATTRIBUTION.md/provenance.json and applicable YOLOv5 license obligations.
Installing a checkpoint or merging/deploying the application requires a separate,
explicit operator decision; no such deployment is performed here.


In [ ]:
import hashlib
from google.colab import files

with BEST.open("rb") as f:
    print("MODEL_SHA256:", hashlib.file_digest(f, "sha256").hexdigest())
files.download(str(BEST))
files.download(str(reports[-1]))
if DATA_SOURCE == "public-pilot":
    files.download(str(DATASET / "provenance.json"))
    files.download(str(DATASET / "ATTRIBUTION.md"))
